In [ ]:
#### GENERAL SCRIPT TEMPLATE for HARVEST and WEIGHTS ####
# maxent_reweight_agroforestry.py
#
# Purpose
#   1) Load weighted edge list at tick T (e.g., 416, 442, 468)
#   2) Compute effective weights: w_eff = w * exp(-(λ_comp*φ_shade + λ_root*φ_root + λ_P*φ_P))
#   3) For each Ilex plant j, compute correction factor C_j = sum_incident(w_eff)/sum_incident(w)
#   4) Apply to harvest at tick h: H_new = H_old * C_j
#   5) Aggregate by “dominant neighbor species” from harvest file (species1_neighbors..species9_neighbors)
#   6) Compute %to-median and species gaps g_s, and optionally grid-search lambdas to reduce |g_s|
#
# Files expected in the working directory:
#   weighted_matrix_with_coords_combined_tick_416.csv
#   weighted_matrix_with_coords_combined_tick_442.csv
#   weighted_matrix_with_coords_combined_tick_468.csv
#   harvest_per_plant_40RepositionVariable.csv
#
# Notes
#   - We treat each harvest row as one Ilex plant j=who.
#   - For C_j, we use *all* edges incident to j in the weight file (source==j or target==j).
#   - Tree species per edge is parsed from the "type" column like "species0-species1".
#     We assume the nonzero species index is the tree (1..9), species0 is Ilex/control.
#
# Install requirements:
#   pip install pandas numpy

from __future__ import annotations
import math
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import os
print("CWD:", os.getcwd())



# ----------------------------
# 1) Field-derived proxies
# ----------------------------

# ΔP2O5 and ΔSOM (2013 -> 2021) per *tree* species index
# Species mapping (NetLogo):
# 1 Toona, 2 Cañafístola, 3 Lapacho, 4 Petiribí, 5 Anchico, 6 Araucaria, 7 Guatambú, 8 Grevillea, 9 Kiri


# Species mapping (NetLogo):
# 1 Toona, 2 Cañafístola, 3 Lapacho, 4 Petiribí, 5 Anchico,
# 6 Araucaria, 7 Guatambú, 8 Grevillea, 9 Kiri
species = ["Toona","Cañafístola","Lapacho","Petiribí","Anchico","Araucaria","Guatambú","Grevillea","Kiri"]
DELTA_SOM = {
    1: 0.00,
    2: 0.33,
    3: 0.43,
    4: -0.30,
    5: 0.72,
    6: -0.46,
    7: 0.98,
    8: 0.30,
    9: 0.43,
}
DELTA_P2O5 = {
    1: 6.28,
    2: 0.71,
    3: 6.06,
    4: 5.46,
    5: 6.56,
    6: 3.10,
    7: 4.26,
    8: 2.26,
    9: 12.43,
}

# Harvest gaps g_s = (%to-median)_field - (%to-median)_sim
# Species-level harvest gap used as fitting target:
# g_s = (field relative performance) - (sim relative performance)
# g_s < 0  => simulation is too low, should move UP
# g_s > 0  => simulation is too high, should move DOWN

GAP_G  = {1: -6.96, 2: 3.88, 3:-8.52, 4:-1.59, 5:8.34, 6:-11.34, 7: 12.06, 8:2.63, 9: -4.69,}  ## (global Field - Sim:Abs value of Net Deviation Field-Sim Global)

# GAP_G = { 1:-17.83, 2:5.93, 3:-39.32, 4:13.54, 5:8.14, 6:10.80, 7:11.51, 8:-0.30, 9:-1.53,}  ##g^ == γ(Abs value of Net Deviation Field-Sim monospecies)




# ----------------------------
# 2) Helpers
# ----------------------------

def zscore_dict(d: Dict[int, float]) -> Dict[int, float]:
    xs = np.array([d[k] for k in sorted(d.keys())], dtype=float)
    mu = xs.mean()
    sd = xs.std(ddof=0)
    if sd == 0:
        return {k: 0.0 for k in d}
    return {k: (d[k] - mu) / sd for k in d}

def read_table_safely(path: str) -> pd.DataFrame:
    # Try comma first, then fallback to auto-detect
    try:
        df = pd.read_csv(path, sep=",")
    except Exception:
        df = pd.read_csv(path, sep=None, engine="python")
    # strip whitespace from column names
    df.columns = [c.strip() for c in df.columns]
    # Try to convert all columns that should be numeric
    for col in df.columns:
        if df[col].dtype == object:
            try:
                # Only convert if all values look like numbers or are empty
                if df[col].str.match(r'^-?\d+(\.\d+)?$', na=True).all():
                    df[col] = pd.to_numeric(df[col], errors='coerce')
                else:
                    # Check for suspicious repeated float patterns (malformed column)
                    skip = False
                    # If any value in the column contains a pattern like '2.150424022.15042402...' (concatenated floats)
                    if df[col].str.contains(r'(\d+\.\d{2,}){2,}', na=False).any():
                        print(f"Warning: Column '{col}' in {path} contains concatenated floats. Skipping conversion.")
                        skip = True
                    # If any value in the column contains more than one float number without a separator
                    if df[col].str.contains(r'(\d+\.\d+){2,}', na=False).any():
                        print(f"Warning: Column '{col}' in {path} contains repeated floats without separator. Skipping conversion.")
                        skip = True
                    # If any value in the column contains more than one '.' and no separator, skip conversion
                    if df[col].apply(lambda x: isinstance(x, str) and x.count('.') > 1 and not any(sep in x for sep in [',', ';', '\t', ' '])).any():
                        print(f"Warning: Column '{col}' in {path} appears malformed (concatenated floats). Skipping conversion.")
                        skip = True
                    if skip:
                        print(f"Column '{col}' skipped from numeric conversion due to detected malformed values.")
                        continue  # <--- SKIP conversion for this column!
                    # Otherwise, do NOT attempt conversion for non-matching columns
                    # (leave as object/string)
            except Exception as e:
                print(f"Warning: Could not convert column '{col}' to numeric: {e}")
    return df


def parse_tree_species_from_type(type_str: str) -> Optional[int]:
    """
    Parse 'species0-species1' -> returns 1 (tree species index).
    If neither side is 'species0', returns the second index by default.
    """
    if not isinstance(type_str, str) or "species" not in type_str:
        return None
    parts = type_str.split("-")
    if len(parts) != 2:
        return None
    def idx(p: str) -> Optional[int]:
        p = p.strip()
        if not p.startswith("species"):
            return None
        try:
            return int(p.replace("species", ""))
        except:
            return None
    a = idx(parts[0]); b = idx(parts[1])
    if a is None or b is None:
        return None
    if a == 0 and b != 0:
        return b
    if b == 0 and a != 0:
        return a
    # if both nonzero, pick b (arbitrary; update if your data implies direction)
    return b

def distance_factor(link_interaction_type: str, eta: float = 0.5) -> float:
    """
    Use combined-1 vs combined-2 (or label) as neighbor shell.
    """
    if not isinstance(link_interaction_type, str):
        return 1.0
    s = link_interaction_type.lower()
    if "combined-2" in s or "distance-2" in s or "shade-2" in s or "soil-2" in s:
        return eta
    return 1.0

def dominant_neighbor_species_from_harvest_row(row: pd.Series) -> int:
    """
    Use the neighbor count columns to assign the dominant tree species around this Ilex plant at harvest tick.
    If all are zero -> 0 (control).
    """
    cols = [f"species{i}_neighbors" for i in range(1, 10)]
    vals = np.array([row.get(c, 0) for c in cols], dtype=float)
    if np.all(vals <= 0):
        return 0
    return int(np.argmax(vals) + 1)  # +1 because species1..species9

def percent_to_median(xs: Dict[int, float]) -> Dict[int, float]:
    """
    Convert species totals to % deviation from median: 100*(x - median)/median
    """
    keys = sorted(xs.keys())
    arr = np.array([xs[k] for k in keys], dtype=float)
    med = np.median(arr)
    if med == 0:
        return {k: 0.0 for k in keys}
    return {k: 100.0 * (xs[k] - med) / med for k in keys}

@dataclass
class Potentials:
    P: Dict[int, float]     # facilitation proxy (ΔP2O5)
    R: Dict[int, float]     # root/SOM proxy (ΔSOM)
    S: Dict[int, float]     # shade residual or explicit shade; start with zeros

def build_potentials(use_zscore: bool = True, shade_mode: str = "residual") -> Potentials:
    P = zscore_dict(DELTA_P2O5) if use_zscore else dict(DELTA_P2O5)
    R = zscore_dict(DELTA_SOM)  if use_zscore else dict(DELTA_SOM)

    if shade_mode == "zero":
        S = {k: 0.0 for k in range(1, 10)}

    elif shade_mode == "residual":
        spp = sorted(GAP_G.keys())
        X = np.column_stack([
            np.ones(len(spp)),
            np.array([P[k] for k in spp], dtype=float),
            np.array([R[k] for k in spp], dtype=float),
        ])
        y = np.array([GAP_G[k] for k in spp], dtype=float)

        beta, *_ = np.linalg.lstsq(X, y, rcond=None)
        resid = y - X @ beta

        mu = resid.mean()
        sd = resid.std(ddof=0)
        if sd == 0:
            S = {k: 0.0 for k in spp}
        else:
            S = {k: (resid[i] - mu) / sd for i, k in enumerate(spp)}

    else:
        raise ValueError(f"Unknown shade_mode: {shade_mode}")

    return Potentials(P=P, R=R, S=S)


# ----------------------------
# 3) Core: compute C_j from weights
# ----------------------------

def compute_Cj_from_weight_file(
    weight_csv: str,
    lambdas: Tuple[float, float, float],  # (λ_comp, λ_root, λ_P)
    pot: Potentials,
    eta: float = 0.5
) -> Dict[int, float]:
    """
    Returns dict: Ilex node id (who) -> correction factor C_j
    """
    lam_comp, lam_root, lam_P = lambdas
    
    df = read_table_safely(weight_csv)

    # Defensive column checks
    required = {"source", "target", "weight", "type", "link-interaction-type"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{weight_csv} missing columns: {missing}")

    # Parse species for each edge and f(d)
    df["tree_species"] = df["type"].apply(parse_tree_species_from_type)
    df["f_d"] = df["link-interaction-type"].apply(lambda s: distance_factor(s, eta=eta))

    # Drop edges where we couldn't infer a tree species (rare)
    df = df.dropna(subset=["tree_species"]).copy()
    df["tree_species"] = df["tree_species"].astype(int)

    # Build φ for each channel from species constants
    df["phi_P"]    = df.apply(lambda r: pot.P.get(int(r["tree_species"]), 0.0) * float(r["f_d"]), axis=1)
    df["phi_root"] = df.apply(lambda r: pot.R.get(int(r["tree_species"]), 0.0) * float(r["f_d"]), axis=1)
    df["phi_shade"]= df.apply(lambda r: pot.S.get(int(r["tree_species"]), 0.0) * float(r["f_d"]), axis=1)

    # Effective weight multiplier
    
    E = lam_comp*df["phi_shade"] + lam_root*df["phi_root"] + lam_P*df["phi_P"]
    E = np.clip(E.astype(float), -3.0, 3.0)
    df["w_eff"] = df["weight"].astype(float) * np.exp(-E)

    # For each endpoint node (source and target), accumulate incident sums
    # We'll treat graph as undirected for C_j
    src = df[["source", "weight", "w_eff"]].rename(columns={"source": "node"})
    tgt = df[["target", "weight", "w_eff"]].rename(columns={"target": "node"})
    inc = pd.concat([src, tgt], ignore_index=True)

    sums = inc.groupby("node", as_index=False).agg(w_sum=("weight", "sum"), weff_sum=("w_eff", "sum"))

    # C_j = weff_sum / w_sum
    sums["C"] = sums["weff_sum"] / sums["w_sum"].replace(0, np.nan)
    sums["C"] = sums["C"].fillna(1.0)

    Cj = {int(r["node"]): float(r["C"]) for _, r in sums.iterrows()}
    
    return Cj


# ----------------------------
# 4) Apply to harvest at a tick and aggregate by dominant species
# ----------------------------

def apply_correction_to_harvest_tick(
    harvest_csv: str,
    tick: int,
    Cj: Dict[int, float]
) -> pd.DataFrame:
    
    dfh = read_table_safely(harvest_csv)
    dfh["tick"] = pd.to_numeric(dfh["tick"], errors="coerce").astype("Int64")

    if "tick" not in dfh.columns or "who" not in dfh.columns or "harvested" not in dfh.columns:
        raise ValueError("harvest file must include columns: tick, who, harvested")

    sub = dfh[dfh["tick"] == tick].copy()
    if sub.empty:
        raise ValueError(f"No rows found for tick={tick} in {harvest_csv}")

    sub["who"] = sub["who"].astype(int)
    sub["C"] = sub["who"].map(Cj).fillna(1.0)
   
    sub["harvested"] = pd.to_numeric(sub["harvested"], errors="coerce")
    sub = sub.dropna(subset=["harvested"])  # optional but recommended
    sub["harvest_new"] = sub["harvested"] * sub["C"].astype(float)

    
    # dominant neighbor tree species assignment
    sub["dom_species"] = sub.apply(dominant_neighbor_species_from_harvest_row, axis=1)
    print("Harvest columns:", dfh.columns.tolist())
    print("Unique ticks sample:", sorted(dfh["tick"].dropna().unique())[:15])

    return sub

def aggregate_species_totals(df_tick: pd.DataFrame) -> Tuple[Dict[int, float], Dict[int, float]]:
    """
    Returns (old_totals, new_totals) for dom_species in {0..9}
    """
    g = df_tick.groupby("dom_species").agg(
        old_total=("harvested", "sum"),
        new_total=("harvest_new", "sum")
    ).reset_index()
    old = {int(r["dom_species"]): float(r["old_total"]) for _, r in g.iterrows()}
    new = {int(r["dom_species"]): float(r["new_total"]) for _, r in g.iterrows()}
    # ensure all species keys exist
    for k in range(0, 10):
        old.setdefault(k, 0.0)
        new.setdefault(k, 0.0)
    return old, new

def compute_species_gap_report(new_totals: Dict[int, float]) -> pd.DataFrame:
    """
    Compare new %to-median vs target gaps (field–sim) on species 1..9.
    (Control species0 is ignored for gap scoring, but still reported.)
    """
    pct = percent_to_median({k: new_totals[k] for k in range(1, 10)})
    # Here pct is the *new simulated* %to-median; we compare it to what we want:
    # target_sim_pct = field_pct - g_s. But we only stored g_s, not field_pct itself.
    # So for diagnostics: we compute "residual gap proxy" = g_s - (change toward field).
    # In a later stage, you can plug field_pct explicitly if desired.
    rows = []
    for s in range(1, 10):
        rows.append({
            "species": s,
            "new_sim_%to_median": pct.get(s, 0.0),
            "target_gap_g_s": GAP_G[s],  # field - old_sim
        })
    return pd.DataFrame(rows)


def compute_species_direction_report(
    old_totals: Dict[int, float],
    new_totals: Dict[int, float]
) -> pd.DataFrame:
    pct_old = percent_to_median({k: old_totals[k] for k in range(1, 10)})
    pct_new = percent_to_median({k: new_totals[k] for k in range(1, 10)})

    rows = []
    for s in range(1, 10):
        po = pct_old.get(s, 0.0)
        pn = pct_new.get(s, 0.0)
        gs = GAP_G[s]
        dp = pn - po

        desired = "UP" if gs < 0 else "DOWN"
        actual = "UP" if dp > 0 else ("DOWN" if dp < 0 else "FLAT")
        correct = (dp * gs < 0)

        rows.append({
            "species": s,
            "old_%to_median": po,
            "new_%to_median": pn,
            "delta_pct": dp,
            "g_s": gs,
            "desired_move": desired,
            "actual_move": actual,
            "correct_direction": correct
        })
    return pd.DataFrame(rows)

# ----------------------------
# 5) End-to-end run for your chosen ticks
# ----------------------------

def run_two_harvests_firstpass(
    harvest_csv: str = "harvest_per_plant_40RepositionVariable.csv",
    weight_416: str  = "weighted_matrix_with_coords_combined_tick_416.csv",
    weight_468: str  = "weighted_matrix_with_coords_combined_tick_468.csv",
    lambdas: Tuple[float, float, float] = (0.029, 0.144, - 0.053),  # (λ_comp, λ_root, λ_P)
    use_zscore_potentials: bool = True,
    eta: float = 0.5
):
    pot = build_potentials(use_zscore=use_zscore_potentials, shade_mode="residual")

    # Map: harvest 440 uses weights 416; harvest 492 uses weights 468
    C416 = compute_Cj_from_weight_file(weight_416, lambdas, pot, eta=eta)
    C468 = compute_Cj_from_weight_file(weight_468, lambdas, pot, eta=eta)

    df440 = apply_correction_to_harvest_tick(harvest_csv, 440, C416)
    df492 = apply_correction_to_harvest_tick(harvest_csv, 492, C468)

    old440, new440 = aggregate_species_totals(df440)
    old492, new492 = aggregate_species_totals(df492)

    rep440 = compute_species_gap_report(new440)
    rep492 = compute_species_gap_report(new492)

    # Simple summary print
    print("\n=== Lambdas (λ_comp, λ_root, λ_P) ===", lambdas)
    print("\n--- Tick 440 totals (species 1..9) ---")
    for s in range(1, 10):
        print(f"s{s}: old={old440[s]:.3f}  new={new440[s]:.3f}")
    print("\n--- Tick 492 totals (species 1..9) ---")
    for s in range(1, 10):
        print(f"s{s}: old={old492[s]:.3f}  new={new492[s]:.3f}")

    print("\n--- Tick 440 %to-median (new) ---")
    print(rep440.to_string(index=False))
    print("\n--- Tick 492 %to-median (new) ---")
    print(rep492.to_string(index=False))

    return {
        "df440": df440, "df492": df492,
        "old440": old440, "new440": new440,
        "old492": old492, "new492": new492,
        "rep440": rep440, "rep492": rep492,
    }


def loss_from_old_new_totals(
    old_totals: Dict[int, float],
    new_totals: Dict[int, float],
    target_gap: Dict[int, float],
    cross_weight: float = 2.0
) -> float:
    """
    Loss for directional correction relative to the median.

    target_gap g_s is interpreted as:
      g_s < 0  => simulation is too low, should move UP
      g_s > 0  => simulation is too high, should move DOWN

    We use %to-median values and penalize:
      1) movement in the wrong direction
      2) failure to cross the median when crossing is needed
    """
    pct_old = percent_to_median({k: old_totals[k] for k in range(1, 10)})
    pct_new = percent_to_median({k: new_totals[k] for k in range(1, 10)})

    loss = 0.0

    for s in range(1, 10):
        gs = float(target_gap[s])
        po = float(pct_old.get(s, 0.0))
        pn = float(pct_new.get(s, 0.0))
        dp = pn - po

        # Primary directional requirement:
        # want dp * gs < 0
        wrong_direction_penalty = max(0.0, dp * gs)

        # Secondary penalty:
        # if old sign disagrees with desired side, reward crossing the median
        # desired side: g<0 => should end above old relative position / move upward
        # more concretely:
        #   g<0 : we prefer pn >= 0 eventually
        #   g>0 : we prefer pn <= 0 eventually
        crossing_penalty = 0.0
        if gs < 0 and pn < 0:
            crossing_penalty = abs(gs) * abs(pn) / 100.0
        elif gs > 0 and pn > 0:
            crossing_penalty = abs(gs) * abs(pn) / 100.0

        # Weight stronger mismatches more
        weight = abs(gs)

        loss += weight * wrong_direction_penalty + cross_weight * crossing_penalty

    return float(loss)

# ----------------------------
# 6) Grid search lambdas (optional, iteration step 2)
# ----------------------------

def grid_search_lambdas(
    harvest_csv: str,
    weight_416: str,
    weight_468: str,
    lam_comp_list: List[float],
    lam_root_list: List[float],
    lam_P_list: List[float],
    use_zscore_potentials: bool = True,
    eta: float = 0.5
):
    pot = build_potentials(use_zscore=use_zscore_potentials, shade_mode="residual")
    
    best = None
    results = []

    for lc in lam_comp_list:
        for lr in lam_root_list:
            for lp in lam_P_list:
                lambdas = (lc, lr, lp)  
                C416 = compute_Cj_from_weight_file(weight_416, lambdas, pot, eta=eta)
                C468 = compute_Cj_from_weight_file(weight_468, lambdas, pot, eta=eta)

                df440 = apply_correction_to_harvest_tick(harvest_csv, 440, C416)
                df492 = apply_correction_to_harvest_tick(harvest_csv, 492, C468)

                old440, new440 = aggregate_species_totals(df440)
                old492, new492 = aggregate_species_totals(df492)

                
                L = (
                loss_from_old_new_totals(old440, new440, GAP_G) +
                loss_from_old_new_totals(old492, new492, GAP_G)
                )

                results.append((L, lambdas))
                if best is None or L < best[0]:
                    best = (L, lambdas)

    results.sort(key=lambda x: x[0])
    print("\nTop 10 lambda sets by loss:")
    for L, lam in results[:10]:
        print(f"loss={L:.3f}  lambdas={lam}")
 
    return best, results
        

if __name__ == "__main__":
    out = run_two_harvests_firstpass(
        harvest_csv="harvest_per_plant_40RepositionVariable.csv",
        weight_416="weighted_matrix_with_coords_combined_tick_416.csv",
        weight_468="weighted_matrix_with_coords_combined_tick_468.csv",
        lambdas=(0.029, 0.144, - 0.053),   # (λ_comp, λ_root, λ_P)
        use_zscore_potentials=True,
        eta=0.5
    )

    # Extract variables from the returned dict
    df440 = out['df440']
    df492 = out['df492']
    old440 = out['old440']
    new440 = out['new440']
    old492 = out['old492']
    new492 = out['new492']
    rep440 = out['rep440']
    rep492 = out['rep492']

    # Compute directional reports (not included in the original function)
    dir440 = compute_species_direction_report(old440, new440)
    dir492 = compute_species_direction_report(old492, new492)

    print("\n--- Tick 440 directional report ---")
    print(dir440.to_string(index=False))
    print("\n--- Tick 492 directional report ---")
    print(dir492.to_string(index=False))

    # Store the extended dict in a variable
    result = {
        "df440": df440, "df492": df492,
        "old440": old440, "new440": new440,
        "old492": old492, "new492": new492,
        "rep440": rep440, "rep492": rep492,
        "dir440": dir440, "dir492": dir492,
    }



In [ ]:
out = run_two_harvests_firstpass(lambdas=(0.30000610351562496, -0.10000610351562499, -0.30000610351562496))

In [9]:
pot = build_potentials(use_zscore=True, shade_mode="residual")
print(pd.DataFrame({
    "species_idx": list(range(1,10)),
    "species": species,
    "g_s": [GAP_G[i] for i in range(1,10)],
    "P_s": [pot.P[i] for i in range(1,10)],
    "R_s": [pot.R[i] for i in range(1,10)],
    "S_s": [pot.S[i] for i in range(1,10)],
}).to_string(index=False))

 species_idx     species    g_s       P_s       R_s       S_s
           1       Toona -17.83  0.329949 -0.622525 -1.014253
           2 Cañafístola   5.93 -1.429661  0.138339  0.111195
           3     Lapacho -39.32  0.260449  0.368903 -2.342223
           4    Petiribí  13.54  0.070904 -1.314219  0.874923
           5     Anchico   8.14  0.418403  1.037541  0.722915
           6   Araucaria  10.80 -0.674640 -1.683122  0.511113
           7    Guatambú  11.51 -0.308186  1.637009  0.793924
           8   Grevillea  -0.30 -0.940003  0.069169 -0.171323
           9        Kiri  -1.53  2.272786  0.368903  0.513729


In [10]:
pot = build_potentials(use_zscore=True, shade_mode="residual")

diag = pd.DataFrame({
    "species_idx": list(range(1,10)),
    "species": species,
    "g_s": [GAP_G[i] for i in range(1,10)],
    "P_s": [pot.P[i] for i in range(1,10)],
    "R_s": [pot.R[i] for i in range(1,10)],
    "S_s": [pot.S[i] for i in range(1,10)],
})

print(diag.to_string(index=False))

 species_idx     species    g_s       P_s       R_s       S_s
           1       Toona -17.83  0.329949 -0.622525 -1.014253
           2 Cañafístola   5.93 -1.429661  0.138339  0.111195
           3     Lapacho -39.32  0.260449  0.368903 -2.342223
           4    Petiribí  13.54  0.070904 -1.314219  0.874923
           5     Anchico   8.14  0.418403  1.037541  0.722915
           6   Araucaria  10.80 -0.674640 -1.683122  0.511113
           7    Guatambú  11.51 -0.308186  1.637009  0.793924
           8   Grevillea  -0.30 -0.940003  0.069169 -0.171323
           9        Kiri  -1.53  2.272786  0.368903  0.513729


In [11]:
def refine_lambdas_iteratively(
    harvest_csv: str,
    weight_416: str,
    weight_468: str,
    init_lambdas: Tuple[float, float, float],
    use_zscore_potentials: bool = True,
    eta: float = 0.5,
    n_rounds: int = 5,
    step0: float = 0.05,
    shrink: float = 0.5
):
    """
    Iteratively refine lambdas by local grid search around the current best point.
    """
    current = init_lambdas
    history = []

    for r in range(n_rounds):
        step = step0 * (shrink ** r)

        lc0, lr0, lp0 = current

        lam_comp_list = [lc0 - step, lc0, lc0 + step]
        lam_root_list = [lr0 - step, lr0, lr0 + step]
        lam_P_list    = [lp0 - step, lp0, lp0 + step]

        best, results = grid_search_lambdas(
            harvest_csv=harvest_csv,
            weight_416=weight_416,
            weight_468=weight_468,
            lam_comp_list=lam_comp_list,
            lam_root_list=lam_root_list,
            lam_P_list=lam_P_list,
            use_zscore_potentials=use_zscore_potentials,
            eta=eta
        )

        best_loss, best_lambdas = best
        history.append({
            "round": r + 1,
            "step": step,
            "best_loss": best_loss,
            "best_lambdas": best_lambdas
        })

        print(f"\nRound {r+1}:")
        print(f"  step = {step}")
        print(f"  best_loss = {best_loss:.6f}")
        print(f"  best_lambdas = {best_lambdas}")

        current = best_lambdas

    return current, history

In [ ]:
best_lambdas, history = refine_lambdas_iteratively(
    harvest_csv="harvest_per_plant_40RepositionVariable.csv",
    weight_416="weighted_matrix_with_coords_combined_tick_416.csv",
    weight_468="weighted_matrix_with_coords_combined_tick_468.csv",
   init_lambdas=(0.29883422851562497, -0.00020141601562498058, -0.39981079101562494),
    use_zscore_potentials=True,
    eta=0.5,
    n_rounds=5,
    step0=0.05,
    shrink=0.5
)

print("\nFinal refined lambdas:", best_lambdas)
print("\nHistory:")
for h in history:
    print(h)

In [ ]:
# Round 5:
#  step = 0.003125
#  best_loss = 770.676982
#  best_lambdas = (0.4000030517578125, -0.2000030517578125, -0.4000030517578125)

#  Final refined lambdas: (0.029, 0.144, -0.053)

# Round 5:
# step = 0.003125
# best_loss = 10.295324
# best_lambdas = (0.029, 0.144, -0.053)

In [2]:
import re

text = """
--- Tick 440 totals (species 1..9) ---
s1: old=236.622  new=249.293
s2: old=283.729  new=274.572
s3: old=250.395  new=261.422
s4: old=278.424  new=294.589
s5: old=340.881  new=339.430
s6: old=333.941  new=354.200
s7: old=293.117  new=253.449
s8: old=296.079  new=290.344
s9: old=272.911  new=285.756

--- Tick 492 totals (species 1..9) ---
s1: old=135.292  new=138.064
s2: old=214.861  new=204.845
s3: old=178.735  new=188.535
s4: old=210.844  new=222.647
s5: old=258.141  new=255.948
s6: old=252.885  new=279.160
s7: old=193.552  new=173.497
s8: old=224.214  new=217.856
s9: old=203.073  new=211.727
"""

for line in text.splitlines():
    line = line.strip()
    if not line:
        print()
        continue
    if line.startswith("---"):
        print(line)
        continue

    m = re.search(r"old=([-\d.]+)\s+new=([-\d.]+)", line)
    if m:
        old_val = m.group(1)
        new_val = m.group(2)
        print(f"{old_val}\t{new_val}")


--- Tick 440 totals (species 1..9) ---
236.622	249.293
283.729	274.572
250.395	261.422
278.424	294.589
340.881	339.430
333.941	354.200
293.117	253.449
296.079	290.344
272.911	285.756

--- Tick 492 totals (species 1..9) ---
135.292	138.064
214.861	204.845
178.735	188.535
210.844	222.647
258.141	255.948
252.885	279.160
193.552	173.497
224.214	217.856
203.073	211.727
